# Model Comparison

Loads saved evaluation results from all three models and produces comparison tables and charts.

**Prerequisites:** run `t5_small.ipynb`, `t5_base.ipynb`, and `mbart.ipynb` first so that `results/` folders are populated.

## 0. Setup

In [ ]:
# Clone repo if needed and install dependencies
!git clone https://github.com/PetraMicanovic/nlp-robot-command-parser.git 2>/dev/null || echo "Repo already present"
%cd nlp-robot-command-parser

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
else:
    print('Local environment — skipping Drive mount.')

In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt

DRIVE_RESULTS = '/content/drive/MyDrive/nlp-robot-command-parser/results'
LOCAL_RESULTS = os.path.join(os.getcwd(), 'results')

if IN_COLAB and os.path.isdir(DRIVE_RESULTS):
    RESULTS_ROOT = DRIVE_RESULTS
else:
    RESULTS_ROOT = LOCAL_RESULTS

print(f'Using results root: {RESULTS_ROOT}')

models = {
    'model': ('t5-small', os.path.join(RESULTS_ROOT, 't5-small')),
    'model_t5_base': ('t5-base', os.path.join(RESULTS_ROOT, 't5_base')),
    'model_mbart': ('mbart', os.path.join(RESULTS_ROOT, 'mbart')),
}

splits = ['simple', 'length', 'addprim_jump', 'addprim_turn_left', 'template_around_right', 'template_jump_around_right', 'template_opposite_right', 'template_right','filler_num0', 'filler_num1', 'filler_num2', 'filler_num3','fewshot_num8_rep1',]

print("Results directories:")
for key, (name, path) in models.items():
    if os.path.isdir(path):
        print(f"{name}: {path}")
    else:
        print("NOT FOUND")



## 1. Exact match across SCAN splits

In [ ]:
rows = []
for model_key, (model_name, results_dir) in models.items():
    for split in splits:
        fpath = os.path.join(results_dir, f'evaluation_{split}.json')
        if not os.path.exists(fpath):
            print(f"Missing: {fpath}")
            continue
        with open(fpath) as f:
            r = json.load(f)
        rows.append({
            'model': model_name,
            'split': split,
            'exact_match': r['exact_match'],
        })

if not rows:
    print('\nNo results loaded- check that evaluation files exist.')
else:    
    df_splits = pd.DataFrame(rows)
    df_pivot = df_splits.pivot(index='split', columns='model', values='exact_match')
    print(df_pivot.to_string())

In [ ]:
if 'df_pivot' in dir() and not df_pivot.empty:
    ax = df_pivot.plot(kind='bar', figsize=(10, 5), rot=30)
    ax.set_title('Exact Match across SCAN splits')
    ax.set_ylabel('Exact Match')
    ax.set_xlabel('Split')
    ax.legend(title='Model')
    ax.set_ylim(0, 1)
    plt.tight_layout()

    out_path = os.path.join(RESULTS_ROOT, 'comparison_splits.png')
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f'Chart saved to {out_path}')

## 2. HuRIC evaluation

In [ ]:
huric_rows = []
for model_key, (model_name, results_dir) in models.items():
    fpath = os.path.join(results_dir, 'evaluation_huric.json')
    if not os.path.exists(fpath):
        print(f"Missing: {fpath}")
        continue
    with open(fpath) as f:
        r = json.load(f)
    huric_rows.append({
        'model': model_name,
        'exact_match': r['exact_match'],
        'n_evaluated': r['n_evaluated'],
    })

if not huric_rows:
    print('No HuRIC results found.')
else:
    df_huric = pd.DataFrame(huric_rows).set_index('model')
    print(df_huric.to_string())

## 3. End-to-end pipeline accuracy (Audio → Text → Actions)

In [ ]:
pipeline_rows = []
for model_key, (model_name, results_dir) in models.items():
    fpath = os.path.join(results_dir, 'evaluation_pipeline.json')
    if not os.path.exists(fpath):
        print(f"Missing: {fpath}")
        continue
    with open(fpath) as f:
        r = json.load(f)
    pipeline_rows.append({
        'model': model_name,
        'with_normalization': r['with_normalization']['exact_match'],
        'without_normalization': r['without_normalization']['exact_match'],
    })

if not pipeline_rows:
    print ('No pipeline results found.')
else:
    df_pipeline = pd.DataFrame(pipeline_rows).set_index('model')
    print(df_pipeline.to_string())

## 4. Error analysis by command length

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(15, 4), sharey=True)

for ax, (model_key, (model_name, results_dir)) in zip(axes, models.items()):
    fpath = os.path.join(results_dir, 'length_analysis.json')
    if not os.path.exists(fpath):
        ax.set_title(f'{model_name}\n(no data)')
        continue
    with open(fpath) as f:
        buckets = json.load(f)
    lengths = list(buckets.keys())
    accuracies = [buckets[l].get('exact_match', 0) for l in lengths]
    ax.bar(lengths, accuracies)
    ax.set_title(model_name)
    ax.set_xlabel('Command length')
    ax.set_ylabel('Exact Match')
    ax.set_ylim(0, 1)
    plt.setp(ax.get_xticklabels(), rotation=45)

plt.suptitle('Error analysis by command length')
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/comparison_length.png', dpi=150)
plt.show()
print("Chart saved to results/comparison_length.png")

## 5. Summary table

In [ ]:
summary_rows = []
for model_key, (model_name, results_dir) in models.items():
    row = {'model': model_name}

    # Simple split as main benchmark
    fpath = os.path.join(results_dir, 'evaluation_simple.json')
    if os.path.exists(fpath):
        with open(fpath) as f:
            row['simple'] = json.load(f)['exact_match']

    # HuRIC
    fpath = os.path.join(results_dir, 'evaluation_huric.json')
    if os.path.exists(fpath):
        with open(fpath) as f:
            row['huric'] = json.load(f)['exact_match']

    # Pipeline with normalization
    fpath = os.path.join(results_dir, 'evaluation_pipeline.json')
    if os.path.exists(fpath):
        with open(fpath) as f:
            r = json.load(f)
        row['pipeline_norm'] = r['with_normalization']['exact_match']

    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index('model')
df_summary.columns = ['SCAN simple', 'HuRIC', 'Pipeline (norm)']
print(df_summary.to_string())